# Projekat - Obrada, anotacija i prepoznavanje teksta sa knjiga

Ovaj projekat se bavi pripremom slikovnog dataset-a, anotacijom OCR-om, treniranjem CRNN modela za prepoznavanje teksta i evaluacijom rezultata. Celokupan tok se sastoji iz četiri glavna fajla: `adjust_dataset.py`, `annotate_dataset.py`, `main.py` i `evaluation.py`.


## 1. adjust_dataset.py

Ovaj fajl služi za pripremu slika iz glavnog dataset-a. Pokreće se samo jednom, da bi se slike transformisale.

- Učitava CSV fajl sa informacijama o slikama (`main_dataset.csv`).
- Svaku sliku učitava pomoću OpenCV (`cv2.imread`).
- Menja rezoluciju slike na veličinu 512x512.
- Rezultat skladišti u `adjusted_dataset` direktorijum.
- Pravi se novi CSV (`adjusted_dataset.csv`) sa putanjama do transformisanih slika i originalnim podacima - naziv knjige i autora.

Ako neka slika ne može da se obradi, preskače se i ispisuje se greška.

## 2. annotate_dataset.py

Ovde se vrši anotacija slika korišćenjem EasyOCR-a:

- Učitava prethodno pripremljen CSV (`adjusted_dataset.csv`).
- Koristi `easyocr.Reader` da detektuje tekst na svakoj slici.
- Poredi detektovani tekst sa nazivom knjige (`name`) radi pronalaženja najrelevantnijeg bounding box-a.
(sa obzirom da su bounding boxovi dosta manji, takođe spaja bliske bounding boxove da bi dobili veće, celovitije tekstove)
- Ako je sličnost iznad zadatog praga, čuva bounding box i prepoznati tekst.
- Rezultat se upisuje nazad u CSV kroz kolone `bounding_boxes` i `text`.

Ako slika ne postoji ili ne sadrži relevantan tekst, preskače se.

## 3. main.py

Ovaj fajl sadrži definiciju dataset klase, CRNN model i trening logiku:

**BookTitleDataset**:
- Učitava CSV sa putanjama i anotacijama.
- Parsira bounding box-ove i kroji sliku samo na relevantni deo.
- Pretvara isečenu sliku u grayscale, normalizuje je i skalira proporcionalno.

**CRNN arhitektura**:
- Kombinacija konvolucionih slojeva i LSTM blokova za sekvencijalnu obradu teksta.
- Izlaz je niz karaktera na osnovu definisanog skupa.

**Trening logika**:
- Definiše DataLoader, normalizaciju i CTC loss.
- U svakoj epohi računa gubitak i optimizuje parametre.
- Na kraju čuva model kao `crnn_text_recognition.pth`.

## 4. evaluation.py

Fajl služi za evaluaciju istreniranog modela:

- Učitava `CRNN` klasu i dataset.
- Učitava sačuvane težine modela (`crnn_text_recognition.pth`).
- Kroz DataLoader prolazi slike i dobija predikcije.
- Dekodira predikovani niz karaktera pomoću `argmax` i uklanjanja ponavljanja.
- Upoređuje predikciju sa stvarnim nazivom (`name`) i računa tačnost.
- Ispisuje rezultate za svaku sliku i ukupnu preciznost.

## 5. Tok celog procesa

1. **Priprema dataset-a** – pokretanjem `adjust_dataset.py` prilagođavamo slike i pravimo CSV za njih.
2. **OCR anotacija** – `annotate_dataset.py` dodaje bounding box-ove i tekstualne anotacije.
3. **Trening modela** – `main.py` koristi anotirani dataset i trenira CRNN.
4. **Evaluacija** – `evaluation.py` meri performanse modela i prikazuje rezultate.

Cilj je automatsko prepoznavanje naslova knjiga sa slika, uz prethodnu pripremu i anotaciju podataka.

# Modeli

U zavisnosti od dataseta, pravljeni su različiti modeli - istreniran nad različitim kategorijama knjiga - grafički noveli i manga, naučno-geografske teme, medicinske knjige, i miks sva tri.

# Evaluacija

Modele smo evaluirali tako što smo gledali koliko tačnih predikcija je dati model imao u test datasetu.
Svaki od modela je imao 0% preciznosti - 0 pogodaka.
Jedini koji je imao neke pogodke je bio manga model sa 300 epoha, mada i to je bila 1% preciznost, gde je treniran nad datasetom od ~800 knjiga.

# Zaključak

CRNN se nije dobro pokazao pri izradi ovog projekta. Tu imam dva zapažanja:

- Sam dataset nije bio dobar sa početka - slike su bile male (približno 250x180), nije bilo bounding boxeva i anotacija
- Korice knjige kao takve se dosta razlikuju - veličina, boja, font, raspored teksta mnogo varira od knjige do knjige. Ta činjenica je mnogo uticala na efikasnost učenja modela.
